In [1]:
import random
import numpy as np
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"Using device: {device}")

Using device: mps


In [2]:
model_name = "textattack/distilbert-base-uncased-MRPC"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)

quantized = False
if device == "cpu":
    try:
        model = torch.quantization.quantize_dynamic(
            model,
            {torch.nn.Linear},
            dtype=torch.qint8
        )
        quantized = True
    except Exception as e:
        print(f"Dynamic quantization unavailable, continuing without it: {e}")

model.to(device)
model.eval()

print(f"Loaded model: {model_name}")
print(f"Number of labels: {model.config.num_labels}")
print(f"Quantized on CPU: {quantized}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Loaded model: textattack/distilbert-base-uncased-MRPC
Number of labels: 2
Quantized on CPU: False


In [3]:
dataset = load_dataset("glue", "mrpc", split="validation")
subset_size = 128
dataset = dataset.select(range(subset_size))

print("Dataset split: glue/mrpc validation")
print(f"Deterministic subset size: {len(dataset)}")
print("Example row:")
print(dataset[0])

Dataset split: glue/mrpc validation
Deterministic subset size: 128
Example row:
{'sentence1': "He said the foodservice pie business doesn 't fit the company 's long-term growth strategy .", 'sentence2': '" The foodservice pie business does not fit our long-term growth strategy .', 'label': 1, 'idx': 9}


In [4]:
batch_size = 32
labels = np.array(dataset["label"])
predictions = []
confidences = []

for start_idx in range(0, len(dataset), batch_size):
    batch = dataset[start_idx:start_idx + batch_size]
    inputs = tokenizer(
        batch["sentence1"],
        batch["sentence2"],
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)
        logits = outputs.logits
        probs = torch.softmax(logits, dim=-1)
        preds = torch.argmax(probs, dim=-1)
        confs = probs.max(dim=-1).values

    predictions.extend(preds.cpu().tolist())
    confidences.extend(confs.cpu().tolist())

predictions = np.array(predictions)
confidences = np.array(confidences)

print(f"Completed inference for {len(predictions)} examples.")

Completed inference for 128 examples.


In [5]:
accuracy = accuracy_score(labels, predictions)
f1 = f1_score(labels, predictions)
cm = confusion_matrix(labels, predictions)
avg_confidence = float(confidences.mean())

print("Fast evaluation metrics:")
print(f"Accuracy          : {accuracy:.4f}")
print(f"F1                : {f1:.4f}")
print(f"Average confidence: {avg_confidence:.4f}")
print("Confusion matrix:")
print(cm)

Fast evaluation metrics:
Accuracy          : 0.8984
F1                : 0.9319
Average confidence: 0.8941
Confusion matrix:
[[26 12]
 [ 1 89]]


In [6]:
print("RESULT SUMMARY")
print(f"model={model_name}")
print("dataset_split=glue/mrpc validation")
print(f"subset_size={len(dataset)}")
print(f"device={device}")
print(f"cpu_dynamic_quantization={quantized}")
print(f"accuracy={accuracy:.4f}")
print(f"f1={f1:.4f}")
print(f"avg_confidence={avg_confidence:.4f}")

RESULT SUMMARY
model=textattack/distilbert-base-uncased-MRPC
dataset_split=glue/mrpc validation
subset_size=128
device=mps
cpu_dynamic_quantization=False
accuracy=0.8984
f1=0.9319
avg_confidence=0.8941
